# Commutants and cosets

The commutant (coset) of a vertex subalgebra $A \subset V$ is

$$\mathrm{Com}(A, V) = \{ v \in V : a_{(n)} v = 0 \text{ for all } a \in A,\ n \ge 0 \}.$$

This notebook shows the commutant features:

* `V.commutant(A)` (alias `V.coset(A)`): `A` is a vertex subalgebra of `V` or a list of generators of one. `S.commutant()` is the commutant of a subalgebra `S` in its ambient algebra.
* Only the generators of `A` are tested. The elements killed by all $a_{(n)}$, $n \ge 0$, of a fixed $v$ form a subalgebra, so membership is a finite test. `A` itself is never built.
* `C.weight_space(W)` is the kernel of the generators' modes on $V_W$, with a canonical basis.
* `C.strong_generators_up_to(W)` gives strong generators found weight by weight. Up to weight `W` every weight space is spanned by their normally ordered monomials. Nothing is claimed above `W`.
* `V.commutant(A, conformal_vector='coset')` (or `C.with_conformal_vector('coset')`) gives the coset Virasoro vector $T_V - T_A$; `A` is then given as a subalgebra constructed with its Virasoro vector `T_A`.
* `C.commutant(max_weight=k)` gives double commutants.

Two classical examples follow:

1. **GKO**: $\mathrm{Com}(L_2(\mathfrak{sl}_2), L_1(\mathfrak{sl}_2) \otimes L_1(\mathfrak{sl}_2))$ is the Ising model $L(1/2, 0)$.
2. **Kazama–Suzuki**: the Heisenberg coset of $V^k(\mathfrak{sl}_2) \otimes bc$ is the $N=2$ superconformal algebra with $c = 3k/(k+2)$, for a generic level $k$.

**Prerequisites:** `01_getting_started.ipynb`, `06_subalgebras_ideals_quotients.ipynb`.

**Contents**

1. GKO: the Ising model as a coset
2. Kazama–Suzuki: the $N = 2$ superconformal algebra

In [1]:
import vertex_algebras as va
from vertex_algebras import *
import os, sys, time
from vertex_algebras.universal_vertex_algebra import UniversalVertexAlgebra
from vertex_algebras.normally_ordered_expressions import NormallyOrderedExpressions
from vertex_algebras.lattice_vertex_algebra import LatticeVertexAlgebra

## 1. GKO: the Ising model as a coset

$L_1(\mathfrak{sl}_2) \otimes L_1(\mathfrak{sl}_2)$ is the lattice vertex algebra of $A_1 \oplus A_1$ (Gram matrix $\mathrm{diag}(2, 2)$). Its Virasoro vector is the sum of the two Sugawara vectors, $T_V = \frac14 :J_0J_0: + \frac14 :J_1J_1:$, with $c = 2$.

The diagonal $\mathfrak{sl}_2$ is generated by $e = e^{\alpha_0} + e^{\alpha_1}$ and $f = e^{-\alpha_0} + e^{-\alpha_1}$. Then $h = e_{(0)} f = J_0 + J_1$ and $e_{(1)} f = 2$, so the level is 2.

In [2]:
L = LatticeVertexAlgebra(QQ, matrix([[2, 0], [0, 2]]))
J0, J1 = L.heisenberg_currents()
L = L.with_conformal_vector(J0.no(J0) / 4 + J1.no(J1) / 4)
J0, J1 = L.heisenberg_currents()
e = L.e((1, 0)) + L.e((0, 1))
f = L.e((-1, 0)) + L.e((0, -1))
h = e.nproduct(f, 0)
h, e.ope(f), h.ope(e), L.central_charge()

(J0 + J1, [J0 + J1, 2], [2*e^(0,1) + 2*e^(1,0)], 2)

The subalgebra generated by $e$ and $f$ is the diagonal $L_2(\mathfrak{sl}_2)$. Its closure adds $h = e_{(0)} f$ as a strong generator. Its Sugawara vector $T_A = \frac{1}{2(k+2)}\left(\frac12 :hh: + :ef: + :fe:\right)$ at $k = 2$ has $c = 3/2$.

In [3]:
A = L.subalgebra([e, f], names=['e', 'f'], conformal_vector=(h.no(h) / 2 + e.no(f) + f.no(e)) / 8)
A.strong_generator_info(), A.central_charge()

([('e', 1, 0, 'given'), ('f', 1, 0, 'given'), ('S0', 1, 0, 'e_(0) f')], 3/2)

The coset. `L.coset(A)`, `L.commutant([e, f])` and `A.commutant()` are the same object, since the commutant only depends on the generators of `A`. Membership is the finite test "all poles of $e(z)v(w)$ and $f(z)v(w)$ vanish", and no weight spaces are needed for it.

In [4]:
C = L.coset(A)
C is L.commutant([e, f]), C is A.commutant(), C

(True,
 True,
 Commutant of the vertex subalgebra generated by [e^(0,1) + e^(1,0), e^(-1,0) + e^(0,-1)] in Lattice vertex algebra of rank 2 with Gram matrix [[2, 0], [0, 2]] over Rational Field)

In [5]:
TV, TA = L.conformal_vector(), A.conformal_vector()
e in C, h in C, TV in C, TV - TA in C

(False, False, False, True)

Weight spaces of the commutant are kernels of the modes $e_{(n)}, f_{(n)}$ ($n \ge 0$) on the weight spaces of $V$. They match the Ising vacuum character, the even part $\frac12\left(\prod_{n \ge 1}(1 + q^{n-1/2}) + \prod_{n \ge 1}(1 - q^{n-1/2})\right)$ of a free Majorana fermion.

In [6]:
N = 9
t0 = time.time()
dims = [C.weight_space(n).dimension() for n in range(N)]
print('weight spaces of the coset: %.1f s' % (time.time() - t0))
P.<t> = PowerSeriesRing(QQ, default_prec=2 * N)         # t = q^(1/2)
majorana = prod(1 + t^(2*n - 1) for n in range(1, N + 1))
ising = [majorana[2 * n] for n in range(N)]
dims, dims == ising

weight spaces of the coset: 0.5 s


([1, 0, 1, 1, 2, 2, 3, 3, 5], True)

The coset Virasoro vector $T_V - T_A$ has $c = 2 - 3/2 = 1/2$. By `strong_generators_up_to`, the coset is strongly generated by a single field of weight 2 (up to weight 8), which is proportional to $T$.

In [7]:
C = C.with_conformal_vector('coset')
T = C.conformal_vector()
gens = C.strong_generators_up_to(8)
C.central_charge(), T, [g.degree() for g in gens]

(1/2,
 -1/4*e^(-1,1) - 1/4*e^(1,-1) + 1/16*:J0 J0: - 1/8*:J0 J1: + 1/16*:J1 J1:,
 [2])

In [8]:
g, = gens
m = T.leading_support()
g * T.coefficient(m) == T * g.coefficient(m)       # g is a multiple of T

True

### The Ising null vector

The universal Virasoro vertex algebra at $c = 1/2$ has dimensions $1, 0, 1, 1, 2, 2, 4, 4, 7, \dots$, but the coset has only 3 states at weight 6. Inside the coset, the four normally ordered monomials of $T$ at weight 6 satisfy one relation, and `express` finds it. This is the singular vector of weight 6 of $V^{1/2}(\mathrm{Vir})$, whose quotient is $L(1/2, 0)$.

In [9]:
S = L.subalgebra([T], names=['T'])
S.print_options(generators='strong')
Ts = S.gen('T')
TTT = Ts.no(Ts.no(Ts))
[S.weight_space(n).dimension() for n in range(N)], S.weight_space(6).basis_words(), TTT

([1, 0, 1, 1, 2, 2, 3, 3, 5],
 [((0, 4),), ((0, 2), (0, 0)), ((0, 1), (0, 1))],
 -93/64*:dT dT: + 33/16*:d^2T T: + 9/128*d^4T)

The same combination in the universal Virasoro algebra at $c = 1/2$ is a singular vector: $L_1 = L_{(2)}$ and $L_2 = L_{(3)}$ kill it. The quotient by the ideal it generates has the dimensions of the coset.

In [10]:
XL = NormallyOrderedExpressions(QQ, ['L'])         # to write the OPE before the algebra exists
l = XL.gen(0)
Vir = UniversalVertexAlgebra(QQ, XL, {('L', 'L'): [l.derivative(), 2*l, 0, 1/4]},
                             degrees={'L': 2})
Lv = Vir.gen('L')
null = Lv.no(Lv.no(Lv)) - (9/128 * Lv.derivative(4) + 33/16 * Lv.derivative(2).no(Lv)
                           - 93/64 * Lv.derivative().no(Lv.derivative()))
Q = Vir / [null]
(Lv.nproduct(null, 2), Lv.nproduct(null, 3),
 [Vir.weight_space(n).dimension() for n in range(N)],
 [Q.weight_space(n).dimension() for n in range(N)] == dims)

(0, 0, [1, 0, 1, 1, 2, 2, 4, 4, 7], True)

### The double commutant

The coset is generated by $T$ (weight 2), so `C.commutant(max_weight=2)` is $\mathrm{Com}(\mathrm{Com}(A, V), V)$. It contains $A$ and has the same dimensions, so the diagonal $L_2(\mathfrak{sl}_2)$ is itself a coset: it is the commutant of the Ising model.

In [11]:
D = C.commutant(max_weight=2)
([D.weight_space(n).dimension() for n in range(6)],
 [A.weight_space(n).dimension() for n in range(6)],
 e in D, f in D, h in D)

([1, 3, 9, 15, 30, 54], [1, 3, 9, 15, 30, 54], True, True, True)

## 2. Kazama–Suzuki: the $N = 2$ superconformal algebra

Take the universal affine vertex algebra $V^k(\mathfrak{sl}_2)$ at a **generic** level, i.e. over $\mathbb{Q}(k)$, tensored with a $bc$ system of fermions of weight $1/2$, $b(z)c(w) \sim 1/(z-w)$. The central charges are $3k/(k+2)$ and $1$.

The fermion number current $J_f = :bc:$ gives $b$ charge $+1$ and $c$ charge $-1$. The diagonal Heisenberg field $H_{\mathrm{tot}} = H + 2J_f$ has level $2k + 4$, and it commutes with $E \otimes c$ and $F \otimes b$.

In [12]:
K.<k> = Frac(QQ['k'])
XS.<E, H, F> = NormallyOrderedExpressions(K)
sl2 = UniversalVertexAlgebra(K, XS, {('H', 'H'): [0, 2*k], ('H', 'E'): [2*E], ('H', 'F'): [-2*F],
                                     ('E', 'F'): [H, k]},
                             conformal_vector=(H*H / 2 + E*F + F*E) / (2*(k + 2)))
E, H, F = sl2.gens()
XB.<b, c> = NormallyOrderedExpressions(K, parities={'b': 1, 'c': 1})
bc = UniversalVertexAlgebra(K, XB, {('b', 'c'): [1]},
                            conformal_vector=(b.derivative()*c - b*c.derivative()) / 2)
b, c = bc.gens()
Jf = b.no(c)
sl2.central_charge(), bc.central_charge(), Jf.ope(Jf), Jf.ope(b), Jf.ope(c)

(3*k/(k + 2), 1, [0, 1], [b], [-c])

In [13]:
V = sl2.tensor(bc, conformal_vector='factors')
Htot = V.embed(0, H) + 2 * V.embed(1, Jf)
U = V.subalgebra([Htot], names=['Htot'], conformal_vector=Htot.no(Htot) / (4 * (k + 2)))
Htot.ope(Htot), V.central_charge(), U.central_charge()

([0, 2*k + 4], (4*k + 2)/(k + 2), 1)

The coset $\mathrm{Com}(\langle H_{\mathrm{tot}} \rangle, V^k(\mathfrak{sl}_2) \otimes bc)$ has central charge $\frac{3k}{k+2} + 1 - 1 = \frac{3k}{k+2}$. Its weight spaces match the vacuum character of the universal $N = 2$ superconformal algebra, generated by $J, G^\pm, T$ of weights $1, \frac32, \frac32, 2$:

$$\prod_{n \ge 1} \frac{1}{1 - q^n} \prod_{n \ge 2} \frac{1}{1 - q^n} \prod_{r \ge 3/2} (1 + q^r)^2 .$$

In [14]:
C2 = V.coset(U, conformal_vector='coset')
M = 10                                                  # weights 0, 1/2, ..., 5
t0 = time.time()
dims2 = [C2.weight_space(n / 2).dimension() for n in range(M + 1)]
print('weight spaces of the coset: %.1f s' % (time.time() - t0))
P.<t> = PowerSeriesRing(QQ, default_prec=M + 1)        # t = q^(1/2)
n2 = (prod(1 / (1 - t^(2*n)) for n in range(1, M)) * prod(1 / (1 - t^(2*n)) for n in range(2, M))
      * prod((1 + t^m)^2 for m in range(3, M + 1, 2)))
C2.central_charge(), dims2, dims2 == [n2[i] for i in range(M + 1)]

weight spaces of the coset: 0.1 s


(3*k/(k + 2), [1, 0, 1, 2, 3, 4, 6, 10, 15, 20, 28], True)

Strong generators up to weight 5: one field of weight 1, two of weight $3/2$ ($F \otimes b$ and $E \otimes c$), and one of weight 2. Nothing new appears up to weight 5. These are the canonical basis vectors, not yet normalised.

In [15]:
J1, Gm1, Gp1, W1 = C2.strong_generators_up_to(5)
[g.degree() for g in C2.strong_generators_up_to(5)], J1, Gm1, Gp1

([1, 3/2, 3/2, 2], 1 # :b c: - 1/k*H # 1, F # b, E # c)

Normalise to the standard $N = 2$ basis: $J$ with $J(z)G^\pm(w) \sim \pm G^\pm(w)/(z-w)$, $T$ the coset Virasoro vector, and $G^-$ rescaled so that

$$G^+(z)G^-(w) \sim \frac{2c/3}{(z-w)^3} + \frac{2J(w)}{(z-w)^2} + \frac{2T(w) + \partial J(w)}{z-w}.$$

All $N = 2$ OPEs then hold exactly, for generic $k$.

In [16]:
T2 = C2.conformal_vector()
cc = C2.central_charge()
m = Gp1.leading_support()
J = J1 / (J1.nproduct(Gp1, 0).coefficient(m) / Gp1.coefficient(m))
Gp = Gp1
Gm = Gm1 * (2*cc/3) / Gp1.ope(Gm1)[2].coefficient(Gp1.ope(Gm1)[2].leading_support())
checks = {
    'J J':   J.ope(J) == [0, cc/3 * C2.one()],
    'J G+':  J.ope(Gp) == [Gp],
    'J G-':  J.ope(Gm) == [-Gm],
    'T J':   T2.ope(J) == [J.derivative(), J],
    'T G+':  T2.ope(Gp) == [Gp.derivative(), 3/2 * Gp],
    'T G-':  T2.ope(Gm) == [Gm.derivative(), 3/2 * Gm],
    'G+ G-': Gp.ope(Gm) == [2*T2 + J.derivative(), 2*J, 2*cc/3 * C2.one()],
    'G+ G+': Gp.ope(Gp) == [],
    'G- G-': Gm.ope(Gm) == [],
}
checks

{'J J': True,
 'J G+': True,
 'J G-': True,
 'T J': True,
 'T G+': True,
 'T G-': True,
 'G+ G-': True,
 'G+ G+': True,
 'G- G-': True}

The subalgebra generated by $J, G^\pm, T$: its closure adds no strong generator, and it prints in the generators. Its dimensions equal those of the coset, so up to weight 5 the coset **is** the universal $N = 2$ superconformal algebra at $c = 3k/(k+2)$.

In [17]:
N2 = V.subalgebra([J, Gp, Gm, T2], names=['J', 'G+', 'G-', 'T'])
N2.print_options(generators='strong')
Jn, Gpn, Gmn, Tn = N2.gens()
print(N2.variable_names())
for x, y in [(Gpn, Gmn), (Gmn, Gpn), (Jn, Gpn), (Tn, Gmn), (Tn, Tn)]:
    print(x, '×', y, ':', x.ope(y))
[N2.weight_space(n / 2).dimension() for n in range(M + 1)] == dims2

('J', 'G+', 'G-', 'T')
G+ × G- : [dJ + 2*T, 2*J, 2*k/(k + 2)]
G- × G+ : [-dJ + 2*T, -2*J, 2*k/(k + 2)]
J × G+ : [G+]
T × G- : [dG-, 3/2*G-]
T × T : [dT, 2*T, 0, 3/2*k/(k + 2)]


True

Elements of the coset can be written in the strong generators found by the search, within the searched weights:

In [18]:
C2.express(Gp1.no(Gm1) + J1.derivative(2))

(-1/2*k + 1)*d^2C0 - :C1 C2: - dC3

### The double commutant

The coset is generated in weights $\le 2$, so `C2.commutant(max_weight=2)` is the double commutant. Its dimensions are $p(n)$ at integer weights and $0$ at half-integer weights, and it contains $H_{\mathrm{tot}}$. So the Heisenberg algebra $\langle H_{\mathrm{tot}} \rangle$ is itself a coset (up to the weight checked).

In [19]:
D2 = C2.commutant(max_weight=2)
[D2.weight_space(n / 2).dimension() for n in range(11)], Htot in D2

([1, 0, 1, 0, 2, 0, 3, 0, 5, 0, 7], True)

## Notes

* Everything above is exact. The coset weight spaces are kernels computed by sparse or dense linear algebra over $\mathbb{Q}$ or $\mathbb{Q}(k)$.
* `strong_generators_up_to(W)` makes no claim above `W`. To get a vertex algebra with ideals and quotients, use `V.subalgebra(C.strong_generators_up_to(W))`, which is contained in the coset; compare dimensions to see equality, as done above.
* For large weight spaces, ordering matters. The ambient basis is ordered by the key order, so the kernel comes out directly as the canonical basis, which usually has small coefficients. For example, the $\mathfrak{sl}_2$ parafermions at weight 14 (ambient dimension 3105) take 27 s.

## References

- [GKO1985] P. Goddard, A. Kent, D. Olive, *Virasoro algebras and coset space models*, Phys. Lett. B 152 (1985), 88–92, [doi:10.1016/0370-2693(85)91145-1](https://doi.org/10.1016/0370-2693(85)91145-1).
- [GKO1986] P. Goddard, A. Kent, D. Olive, *Unitary representations of the Virasoro and super-Virasoro algebras*, Comm. Math. Phys. 103 (1986), 105–119, [doi:10.1007/BF01464283](https://doi.org/10.1007/BF01464283). — the coset construction and the Ising model as a coset (section 1).
- [KS1989] Y. Kazama, H. Suzuki, *New N = 2 superconformal field theories and superstring compactification*, Nuclear Phys. B 321 (1989), 232–268, [doi:10.1016/0550-3213(89)90250-2](https://doi.org/10.1016/0550-3213(89)90250-2). — the $N=2$ superconformal algebra as a coset (section 2).
- [BPZ1984] A. A. Belavin, A. M. Polyakov, A. B. Zamolodchikov, *Infinite conformal symmetry in two-dimensional quantum field theory*, Nuclear Phys. B 241 (1984), 333–380, [doi:10.1016/0550-3213(84)90052-X](https://doi.org/10.1016/0550-3213(84)90052-X). — the Ising model.